<a href="https://colab.research.google.com/github/adrianmwale/bank-churn-analysis/blob/main/bank_churn_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("Churn_Modelling.csv")

con = sqlite3.connect("churn.db")
df.to_sql("customers", con, index=False, if_exists="replace")

print(len(df), "rows loaded into table 'customers'")

10000 rows loaded into table 'customers'


In [2]:
query = """
SELECT
    COUNT(*)                        AS customers,
    SUM(Exited)                     AS left_bank,
    ROUND(100.0 * AVG(Exited), 1)   AS churn_pct
FROM customers;
"""
pd.read_sql(query, con)

,customers,left_bank,churn_pct
0,10000,2037,20.4


In [3]:
query = """
SELECT
    Geography,
    COUNT(*)                        AS customers,
    ROUND(100.0 * AVG(Exited), 1)   AS churn_pct
FROM customers
GROUP BY Geography
ORDER BY churn_pct DESC;
"""
pd.read_sql(query, con)

,Geography,customers,churn_pct
0,Germany,2509,32.4
1,Spain,2477,16.7
2,France,5014,16.2


**Germany has the highest churn percentage at 32.4%, which is about double the rate in Spain (16.7%) and France (16.2%)**

In [4]:
query = """
SELECT
    NumOfProducts,
    COUNT(*)                        AS customers,
    ROUND(100.0 * AVG(Exited), 1)   AS churn_pct
FROM customers
GROUP BY NumOfProducts
ORDER BY NumOfProducts;
"""
pd.read_sql(query, con)

,NumOfProducts,customers,churn_pct
0,1,5084,27.7
1,2,4590,7.6
2,3,266,82.7
3,4,60,100.0


**Churn by number of products shows a sharp divide: customers with 2 products churn at just 7.6%, but the rate jumps to 82.7% for those with 3 products (266 customers) and reaches 100% for those with 4 products (60 customers). Note, however, that the 3- and 4-product groups are quite small.**

In [5]:
query = """
SELECT
    CASE
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 40 THEN '30-39'
        WHEN Age < 50 THEN '40-49'
        WHEN Age < 60 THEN '50-59'
        ELSE '60+'
    END                             AS age_band,
    COUNT(*)                        AS customers,
    ROUND(100.0 * AVG(Exited), 1)   AS churn_pct
FROM customers
GROUP BY age_band
ORDER BY MIN(Age);
"""
pd.read_sql(query, con)

,age_band,customers,churn_pct
0,18-29,1641,7.6
1,30-39,4346,10.9
2,40-49,2618,30.8
3,50-59,869,56.0
4,60+,526,27.9


**Churn increases with age from 7.6% in the 18–29 group to 10.9% (30–39), 30.8% (40–49), and peaks at 56.0% in the 50–59 band, before falling to 27.9% for those aged 60+. So while the earlier claim that older customers leave more holds true up to the 50s, the 60+ group bucks the trend. The 50–59 band is the highest-risk segment.**

In [6]:
query = """
SELECT
    IsActiveMember,
    COUNT(*)                        AS customers,
    ROUND(100.0 * AVG(Exited), 1)   AS churn_pct
FROM customers
GROUP BY IsActiveMember;
"""
pd.read_sql(query, con)

,IsActiveMember,customers,churn_pct
0,0,4849,26.9
1,1,5151,14.3


**Inactive members(26.9%) churn at roughly double the rate of active ones(14.3%). Since inactive customers make up about half the base (4,849 of 10,000), this represents a substantial segment.**